# AUG-PE: Differentially Private Synthetic Text via Foundation Model APIs

This notebook walks through the AUG-PE algorithm step by step,
importing directly from the original codebase.

**Prerequisites**: Run `bash scripts/local_scripts/install_gpu.sh` first.

In [1]:
import os, sys
import numpy as np
import collections

# Ensure repo root is on the path
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

print(f'Working directory: {os.getcwd()}')

Working directory: /home/ubuntu/pe_llm_synthetic_gen


## 1. Privacy Accounting

Before running any experiment, verify the DP noise multipliers from the paper.

In [2]:
from src.dp_accounting import compute_sigma, compute_epsilon, compute_delta_default

# Yelp: n_priv = 1,939,290
n_priv = 1_939_290
delta = compute_delta_default(n_priv)
T = 10

print(f'Yelp: n_priv={n_priv:,}, delta={delta:.2e}, T={T}')
print()

for eps_target in [1.0, 2.0, 4.0]:
    sigma = compute_sigma(eps_target, T, delta)
    eps_check = compute_epsilon(sigma, T, delta)
    print(f'  epsilon={eps_target:.1f} => sigma={sigma:.2f} (verify: eps={eps_check:.4f})')

Yelp: n_priv=1,939,290, delta=3.56e-08, T=10

  epsilon=1.0 => sigma=15.40 (verify: eps=1.0000)
  epsilon=2.0 => sigma=8.04 (verify: eps=2.0000)
  epsilon=4.0 => sigma=4.25 (verify: eps=4.0000)


## 2. Load Private Data

Load the Yelp dataset and examine its label distribution.

In [3]:
from src.dpsda.data_loader import load_data

train_data, train_labels, label_counter, label_indexer = load_data(
    dataset='yelp',
    data_file='data/yelp/train.csv',
    num_samples=5000,  # subsample for this demo
)

print(f'Loaded {len(train_data)} private samples')
print(f'Number of label combinations: {len(label_counter)}')
print()
print('Top 10 label combinations:')
for label, count in label_counter.most_common(10):
    print(f'  {label}: {count}')

data_file data/yelp/train.csv
[ 633607 1907444 1257699 ...  535964 1251197  558984]
Loaded 4999 private samples
Number of label combinations: 50

Top 10 label combinations:
  Business Category: Restaurants	Review Stars: 5.0: 1316
  Business Category: Restaurants	Review Stars: 4.0: 712
  Business Category: Restaurants	Review Stars: 1.0: 388
  Business Category: Restaurants	Review Stars: 3.0: 370
  Business Category: Restaurants	Review Stars: 2.0: 271
  Business Category: Bars	Review Stars: 5.0: 192
  Business Category: Beauty & Spas	Review Stars: 5.0: 167
  Business Category: Bars	Review Stars: 4.0: 136
  Business Category: Shopping	Review Stars: 5.0: 128
  Business Category: Event Planning & Services	Review Stars: 5.0: 114


In [4]:
# Show a few examples
print('--- Sample private texts ---')
for i in range(3):
    print(f'\n[{train_labels[i]}]')
    print(train_data[i][:200] + '...' if len(train_data[i]) > 200 else train_data[i])

--- Sample private texts ---

[Business Category: Health & Medical	Review Stars: 1.0]
First off, I am no stranger to chiropractic care. In fact I've gotten hundreds of adjustments at a number of places. Some good and some mediocre.  But I've never had a bad experience like I had at thi...

[Business Category: Event Planning & Services	Review Stars: 5.0]
This is a great place for Barbq. Grab some pickles and peppers when you walk in at the bar. I love the ribs and the beef. The sandwiches are generous, and the sweet tea keeps flowing. The sauce is the...

[Business Category: Restaurants	Review Stars: 5.0]
Outstanding! Amazing food and the service was very professional. Highly recommended!!! Foie Gras and Duck were both incredible. Probably the best food that we have had in the region so far. Worth brin...


## 3. Compute Private Embeddings

Use the sentence-transformer model to embed the private data (Algorithm 1, Line 1).

In [5]:
from src.dpsda.feature_extractor import extract_features

EMBEDDING_MODEL = 'stsb-roberta-base-v2'

print(f'Computing embeddings with {EMBEDDING_MODEL}...')
private_features = extract_features(
    data=train_data,
    batch_size=1024,
    model_name=EMBEDDING_MODEL,
)
print(f'Private embeddings shape: {private_features.shape}')

Computing embeddings with stsb-roberta-base-v2...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: sentence-transformers/stsb-roberta-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 5/5 [00:05<00:00,  1.07s/it]

Private embeddings shape: (4999, 768)


## 4. RANDOM_API: Generate Initial Synthetic Samples

Use GPT-2 with category/rating prompts to generate initial samples (Algorithm 1, Line 2).

In [6]:
from src.apis.hf_api import HFAPI

# Instantiate the HuggingFace GPT-2 API
# Using small batch size and few samples for this demo
api = HFAPI(
    model_type='gpt2',
    variation_type='yelp_rephrase_tone',
    use_subcategory=True,
    output_dir=None,
    seed=42,
    mlm_probability=0.5,
    length=64,
    temperature=1.4,
    top_k=50,
    top_p=0.9,
    repetition_penalty=1.0,
    do_sample=True,
    fp16=True,
    no_cuda=False,
    random_sampling_batch_size=64,
    num_beams=5,
    dry_run=False,
    variation_batch_size=64,
)
print('GPT-2 model loaded.')

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT-2 model loaded.


In [7]:
# Generate a small set of initial samples (Nsyn_demo samples)
Nsyn_demo = 100  # small for demo; paper uses 5000

# Scale down the label counter proportionally
demo_counter = collections.Counter()
total = sum(label_counter.values())
for label, count in label_counter.items():
    demo_count = max(1, round(count / total * Nsyn_demo))
    demo_counter[label] = demo_count

print(f'Generating {sum(demo_counter.values())} initial samples...')
initial_samples, initial_labels, sync_counter, all_prompts = api.text_random_sampling(
    num_samples=Nsyn_demo,
    prompt_counter=label_counter,
)
print(f'Generated {len(initial_samples)} initial samples')
print()
print('--- Sample generated texts ---')
for i in range(min(3, len(initial_samples))):
    print(f'\n[{initial_labels[i]}]')
    text = initial_samples[i]
    print(text[:200] + '...' if len(text) > 200 else text)

Generating 115 initial samples...


  0%|          | 0/50 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
  2%|▏         | 1/50 [00:01<01:02,  1.28s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
  4%|▍         | 2/50 [00:02<01:04,  1.35s/it]Th

Generated 94 initial samples

--- Sample generated texts ---

[Business Category: Health & Medical	Review Stars: 1.0]
for the Sick

[Business Category: Event Planning & Services	Review Stars: 5.0]
Rating: 9/10 This is the best way to stay up to date on news and new information related to the Event Calendar. Get updated as often as possible as you are working through this important and complex i...

[Business Category: Event Planning & Services	Review Stars: 5.0]
: 10-20 Event Status Date Posted: Friday, August 27, 2015 Author Review: Not currently editing the review The event that has generated such interest and interest on this site is Event: Events hosted b...


## 5. One PE Iteration

Run the core loop of AUG-PE once: embed synthetic samples, compute DP histogram,
select top samples, generate variations.

In [8]:
from src.dpsda.dp_counter import dp_nn_histogram

# Step 5a: Embed synthetic samples (K=0, self-embedding)
print('Computing synthetic embeddings...')
syn_features = extract_features(
    data=initial_samples,
    batch_size=1024,
    model_name=EMBEDDING_MODEL,
)
print(f'Synthetic embeddings shape: {syn_features.shape}')

Computing synthetic embeddings...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: sentence-transformers/stsb-roberta-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 1/1 [00:00<00:00, 10.48it/s]

Synthetic embeddings shape: (94, 768)


In [9]:
# Step 5b: DP Nearest Neighbor Histogram (one class at a time)
# For demo: use sigma=0 (non-private) to see clean signal
sigma = 0.0

private_classes = list(label_counter.keys())
all_counts = np.zeros(len(initial_samples))

current_idx = 0
for cls_label in private_classes:
    n_cls = sync_counter.get(cls_label, 0)
    if n_cls == 0:
        continue
    
    cls_syn_features = syn_features[current_idx:current_idx + n_cls]
    cls_pri_indices = label_indexer[cls_label]
    cls_pri_features = private_features[cls_pri_indices]
    
    count, clean_count = dp_nn_histogram(
        public_features=cls_syn_features,
        private_features=cls_pri_features,
        noise_multiplier=sigma,
    )
    all_counts[current_idx:current_idx + n_cls] = count
    current_idx += n_cls

print(f'Histogram: {len(all_counts)} bins, sum={all_counts.sum():.0f}')
print(f'Non-zero bins: {(all_counts > 0).sum()}')
print(f'Top 5 vote counts: {sorted(all_counts, reverse=True)[:5]}')

Histogram: 94 bins, sum=4729
Non-zero bins: 83
Top 5 vote counts: [np.float64(398.0), np.float64(213.0), np.float64(198.0), np.float64(194.0), np.float64(180.0)]


In [10]:
# Step 5c: Rank-based selection (select top samples)
# For AUG-PE with L>1: select top Nsyn/L samples, then generate L-1 variations
L = 2  # small for demo; paper uses L=7
selected_size = len(initial_samples) // L

sort_indices = np.argsort(-all_counts)
selected_indices = sort_indices[:selected_size]

selected_samples = [initial_samples[i] for i in selected_indices]
selected_labels = [initial_labels[i] for i in selected_indices]

print(f'Selected {len(selected_samples)} samples (top by histogram votes)')
print(f'Vote range of selected: [{all_counts[selected_indices[-1]]:.0f}, {all_counts[selected_indices[0]]:.0f}]')

Selected 47 samples (top by histogram votes)
Vote range of selected: [40, 398]


In [11]:
# Step 5d: VARIATION_API - generate paraphrased variations
print(f'Generating {L-1} variation(s) per selected sample...')
variations, var_labels, _, _, _ = api.text_variation(
    sequences=selected_samples,
    additional_info=selected_labels,
    num_variations_per_sequence=L - 1,
    variation_degree=0.5,
)

print(f'Variations shape: {variations.shape}')
print()
print('--- Original vs Variation ---')
for i in range(min(2, len(selected_samples))):
    print(f'\nOriginal [{selected_labels[i]}]:')
    print(f'  {selected_samples[i][:150]}')
    print(f'Variation:')
    print(f'  {variations[i, 0][:150]}')

Generating 1 variation(s) per selected sample...


  0%|          | 0/1 [00:00<?, ?it/s]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
100%|██████████| 1/1 [00:01<00:00,  1.34s/it]

Variations shape: (47, 1)

--- Original vs Variation ---

Original [Business Category: Restaurants	Review Stars: 5.0]:
  Reviews - 12 stars 2.00 2 restaurants 2 reviews 1 reviews 10 ratings 5 stars 10 ratings 0 ratings 1 ratings Great value at no extra price. Great servi
Variation:
  You do not have the time or the inclination in your life, especially not for a low-class guy like me but I'm here. , you do want to know who you are, 

Original [Business Category: Restaurants	Review Stars: 1.0]:
  : None Location: No Rating of Reviews by restaurant Restaurant Name Restaurant Rank Location Restaurant Rating Category Restaurant Service Rank Qualit
Variation:
  Tried: This is the first location from which it is called on the reservation I've been using it with friends, friends and in our area. - It is fairly 


## 6. FID Measurement

Compute the Frechet Inception Distance between the synthetic and private embedding distributions.

In [12]:
from src.dpsda.metrics import calculate_fid

# FID of initial samples vs private data
fid_initial = calculate_fid(syn_features, private_features)
print(f'FID (initial random samples vs private): {fid_initial:.2f}')

# FID of selected samples vs private data
selected_features = extract_features(
    data=selected_samples,
    batch_size=1024,
    model_name=EMBEDDING_MODEL,
)
fid_selected = calculate_fid(selected_features, private_features)
print(f'FID (after 1 PE iteration, selected): {fid_selected:.2f}')
print(f'FID improvement: {fid_initial - fid_selected:.2f}')

FID (initial random samples vs private): 168.86


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: sentence-transformers/stsb-roberta-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 1/1 [00:00<00:00, 19.59it/s]


FID (after 1 PE iteration, selected): 197.32
FID improvement: -28.45


## 7. Next Steps

This demo showed one iteration of AUG-PE. The method produces **differentially private (DP) synthetic data** when you add Gaussian noise to the nearest-neighbor histogram; the noise level is set by `--noise_multiplier` (sigma).

- **With DP (formal privacy):** set `--noise_multiplier` to the sigma from `src.dp_accounting` for your target (ε, δ). For Yelp with ε=1 and T=10 iterations, sigma ≈ 15.34. Use fewer iterations (e.g. T=10) when running with DP.
- **Without DP (baseline):** use `--noise_multiplier 0` for a non-private baseline (faster, no formal guarantee).

Full experiment:

```bash
# Precompute full embeddings (one-time)
bash scripts/embeddings.sh --yelp

# DP run: (ε,δ)-DP with epsilon=1 (set noise_multiplier; use 10 iterations)
export CUDA_VISIBLE_DEVICES=0
# Edit scripts/hf/yelp/generate.sh: set noise=15.34, epochs=10, then:
bash scripts/hf/yelp/generate.sh

# Or run directly with DP parameters:
python src/main.py --train_data_file data/yelp/train.csv --api HFGPT --dataset yelp \
  --noise_multiplier 15.34 --model_type gpt2 --epochs 10 \
  --do_sample --length 64 --fp16 --temperature 1.4 --select_syn_mode rank \
  --num_samples_schedule 35000 --combine_divide_L 7 --init_combine_divide_L 7 \
  --variation_degree_schedule 0.5 --lookahead_degree 0 --use_subcategory \
  --feature_extractor stsb-roberta-base-v2 --feature_extractor_batch_size 1024 \
  --mlm_probability 0.5 --variation_type yelp_rephrase_tone \
  --result_folder result/yelp_dp_eps1 \
  --train_data_embeddings_file result/embeddings/stsb-roberta-base-v2/yelp_train_all.embeddings.npz \
  --random_sampling_batch_size 1024 --variation_batch_size 1024

# Evaluate downstream accuracy (edit result_folder in the script if you used a different path)
bash scripts/hf/yelp/downstream.sh
```

All source code lives under `src/`. See `src/config.py` for the paper's hyperparameter defaults and `src/dp_accounting.py` for privacy budget (sigma ↔ epsilon).